In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
import torch

set_seed(42)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "distilgpt2"   # modelo pequeño y rápido
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)

prompt = "En el futuro cercano, la IA transformará la educación porque"

inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

# 1) Greedy decoding (determinista)
out_ids = model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=False,         # greedy
    eos_token_id=tokenizer.eos_token_id
)
print("== Greedy ==")
print(tokenizer.decode(out_ids[0], skip_special_tokens=True))

# 2) Beam search (más exploración controlada)
out_ids = model.generate(
    **inputs,
    max_new_tokens=60,
    num_beams=3,
    early_stopping=True,
    no_repeat_ngram_size=3,
    eos_token_id=tokenizer.eos_token_id
)
print("\n== Beam search (3 beams) ==")
print(tokenizer.decode(out_ids[0], skip_special_tokens=True))

# 3) Top-k sampling (aleatoriedad acotada)
out_ids = model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=True,
    top_k=50,
    temperature=0.9,
    eos_token_id=tokenizer.eos_token_id
)
print("\n== Top-k (k=50, temp=0.9) ==")
print(tokenizer.decode(out_ids[0], skip_special_tokens=True))

# 4) Nucleus sampling (top-p)
out_ids = model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=True,
    top_p=0.92,              # masa de prob acumulada
    temperature=0.8,
    eos_token_id=tokenizer.eos_token_id
)
print("\n== Top-p (p=0.92, temp=0.8) ==")
print(tokenizer.decode(out_ids[0], skip_special_tokens=True))

# 5) Control de repetición y longitud
out_ids = model.generate(
    **inputs,
    max_new_tokens=80,
    do_sample=True,
    top_p=0.9,
    repetition_penalty=1.2,  # penaliza repetir tokens
    no_repeat_ngram_size=3,  # evita n-gramas repetidos
    eos_token_id=tokenizer.eos_token_id
)
print("\n== Control de repetición (penalty=1.2, no_repeat_ngram_size=3) ==")
print(tokenizer.decode(out_ids[0], skip_special_tokens=True))

# 6) Secuencias de parada (stop sequences) sencillas
#   Cortamos manualmente si aparece "\n\n" en el texto generado.
out_ids = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    top_k=40,
    temperature=0.8,
    eos_token_id=tokenizer.eos_token_id
)
text = tokenizer.decode(out_ids[0], skip_special_tokens=True)
stop_idx = text.find("\n\n")
if stop_idx != -1:
    text = text[:stop_idx]
print("\n== Con 'stop sequence' (\\n\\n) ==")
print(text)


C:\Users\Nil\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Nil\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Nil\.cache\huggingface\hub\models--distilgpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache

== Greedy ==
En el futuro cercano, la IA transformará la educación porque de la educación de la educación de la educación de la educación de la educación de la educación de la educación de la educación de la educación de la educación de la educación de la educación


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



== Beam search (3 beams) ==
En el futuro cercano, la IA transformará la educación porque en el futuridad.


Advertisements
Share this: Print

Email

Twitter

Facebook

Pinterest

LinkedIn

Reddit

Like this: Like Loading...


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



== Top-k (k=50, temp=0.9) ==
En el futuro cercano, la IA transformará la educación porque formarado a confederación de los tienes de cercioso de la orientación.


Lamento en las juego más en los echilibres así aí el estenado para lálo su a tous


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



== Top-p (p=0.92, temp=0.8) ==
En el futuro cercano, la IA transformará la educación porque el agaría cercano.


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



== Control de repetición (penalty=1.2, no_repeat_ngram_size=3) ==
En el futuro cercano, la IA transformará la educación porque que tener los unidos de no selviras. Año essebre con siempre en achurro y quían día hicca como puede estando su nuevo ancien acún asunciaciones informados para sus fotos (susagudos) contribues emporado del vistador

== Con 'stop sequence' (\n\n) ==
En el futuro cercano, la IA transformará la educación porque la educación de la educación del pérez en la educación en el educación de el educación del pérez en la educación del pérez en la educación del pérez en el educación del pérez en la educación del pérez en la educación de el educación del pérez en la educación del pérez en la educación del pérez
